# Genesis 強化学習（Go2 歩行）on Google Colab

参考: https://tototototototo7.hatenablog.com/entry/2025/genesis_test_google_colab

- 学習・評価・動画作成まですべて Colab 上で行い、ローカルには何もインストールしない
- 事前に VS Code の Colab 拡張で **GPU（T4 など）ランタイム** に接続しておく
- ランタイムが切れると環境は消えるので、毎回上から実行し直す

In [ ]:
# GPU が割り当てられているか確認
!nvidia-smi

## 1. インストール

「セッションを再起動」を求められたら、カーネルを再起動してから次のセルに進む。

In [ ]:
!pip install -q genesis-world

In [ ]:
import importlib.metadata as md
import os
import re

GS_VER = md.version('genesis-world')
print('genesis-world', GS_VER)

# インストールした Genesis と同じバージョンのサンプルコードを取得（タグがなければ最新版）
GENESIS_DIR = '/content/Genesis'
if not os.path.exists(GENESIS_DIR):
    repo = 'https://github.com/Genesis-Embodied-AI/Genesis.git'
    if os.system(f'git clone -q --depth 1 --branch v{GS_VER} {repo} {GENESIS_DIR}') != 0:
        os.system(f'git clone -q --depth 1 {repo} {GENESIS_DIR}')
LOCO_DIR = f'{GENESIS_DIR}/examples/locomotion'

# go2_train.py が要求する rsl_rl のバージョン（例: >=5.0.0）を読み取ってインストール
m = re.search(r"rsl-rl-lib\s*([<>=!~]=?\s*[0-9.]+)", open(f'{LOCO_DIR}/go2_train.py').read())
RSL_REQ = f'rsl-rl-lib{m.group(1)}' if m else 'rsl-rl-lib'
print('install', RSL_REQ)
!pip install -q "{RSL_REQ}" tensordict tensorboard

In [ ]:
# Colab では EGL が NVIDIA GPU を見つけるための設定ファイルがなく、描画の初期化で Segmentation fault になる
# （MuJoCo の公式 Colab チュートリアルと同じ対処）
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')
os.environ['PYOPENGL_PLATFORM'] = 'egl'  # 学習スクリプト（子プロセス）にも引き継がれる
!ldconfig -p | grep -i egl_nvidia

## 2. Google Drive（任意）

学習結果を残したい場合のみ `USE_DRIVE = True` にする。

In [ ]:
USE_DRIVE = True
SAVE_DIR = None

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/pai_finalwork'
    os.makedirs(SAVE_DIR, exist_ok=True)

## 3. 学習（PPO）

結果は `examples/locomotion/logs/{EXP_NAME}` に保存される。

In [ ]:
EXP_NAME = 'go2-walking'
NUM_ENVS = 4096       # 並列環境数（GPU メモリが足りなければ減らす）
MAX_ITER = 300        # 学習イテレーション数

%cd {LOCO_DIR}
# -X faulthandler: 異常終了（Segmentation fault）したときに、どこで落ちたかを表示する
!python -X faulthandler go2_train.py -e {EXP_NAME} -b {NUM_ENVS} --max-iterations {MAX_ITER}
!ls {LOCO_DIR}/logs/{EXP_NAME}

In [ ]:
# 学習結果を Drive にコピー
import shutil

if SAVE_DIR:
    shutil.copytree(f'{LOCO_DIR}/logs/{EXP_NAME}', f'{SAVE_DIR}/logs/{EXP_NAME}', dirs_exist_ok=True)
    print('saved to', f'{SAVE_DIR}/logs/{EXP_NAME}')

## 4. 評価（動画で確認）

Colab には画面がないのでビューアは使わず、カメラでオフスクリーン描画して mp4 に保存する。
`gs.init()` はカーネルごとに 1 回しか呼べないため、再実行するときはカーネルを再起動する。

In [ ]:
os.environ['PYOPENGL_PLATFORM'] = 'egl'  # genesis を import する前に設定する

import glob
import pickle
import sys

import torch
import genesis as gs
from rsl_rl.runners import OnPolicyRunner

sys.path.insert(0, LOCO_DIR)
from go2_env import Go2Env

log_dir = f'{LOCO_DIR}/logs/{EXP_NAME}'
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f'{log_dir}/cfgs.pkl', 'rb'))
reward_cfg['reward_scales'] = {}

# gs.init() と環境の作成はカーネルごとに 1 回だけ。2 回目以降はモデルの読み込みだけやり直す
if 'env' not in globals():
    gs.init(backend=gs.gpu, logging_level='warning')

    # Go2Env は内部でシーンを build してしまうので、build の直前にカメラを追加する
    cam = None
    _orig_build = gs.Scene.build

    def _build_with_camera(self, *args, **kwargs):
        global cam
        cam = self.add_camera(res=(640, 480), pos=(2.0, -2.0, 1.5), lookat=(0.0, 0.0, 0.3), fov=40, GUI=False)
        return _orig_build(self, *args, **kwargs)

    gs.Scene.build = _build_with_camera
    env = Go2Env(
        num_envs=1,
        env_cfg=env_cfg,
        obs_cfg=obs_cfg,
        reward_cfg=reward_cfg,
        command_cfg=command_cfg,
        show_viewer=False,
    )
    gs.Scene.build = _orig_build

# 一番新しいチェックポイントを読み込む
ckpt_path = max(glob.glob(f'{log_dir}/model_*.pt'), key=lambda p: int(re.findall(r'model_([0-9]+)', p)[0]))
print('load', ckpt_path)
runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)
runner.load(ckpt_path)
policy = runner.get_inference_policy(device=gs.device)

In [ ]:
# 録画（このセルはカーネルを再起動せずに何度でも実行できる）
NUM_STEPS = 250  # 0.02 秒 x 250 = 5 秒
VIDEO_PATH = '/content/go2_eval.mp4'

obs = env.reset()
# Genesis 1.4 以降は録画中、シミュレーションを進めるたびに自動でフレームが記録される（cam.render() は不要）
cam.start_recording(save_to_filename=VIDEO_PATH, fps=int(1 / env.dt))
with torch.no_grad():
    for _ in range(NUM_STEPS):
        # ロボットを追いかけるようにカメラを動かす
        x, y, _z = env.base_pos[0].cpu().numpy()
        cam.set_pose(pos=(x + 2.0, y - 2.0, 1.5), lookat=(x, y, 0.3))
        actions = policy(obs)
        obs, rews, dones, infos = env.step(actions)
cam.stop_recording()
print('saved', VIDEO_PATH)

In [ ]:
# 動画を表示（出力は ipynb に埋め込まれるので、見終わったら出力をクリアするとファイルが軽くなる）
from IPython.display import Video

if SAVE_DIR:
    shutil.copy(VIDEO_PATH, SAVE_DIR)
Video(VIDEO_PATH, embed=True, width=640)

## 5. バク転

公式の `go2_backflip.py` には学習済みモデル（お手本）しか入っておらず、学習コードがないので、報酬と学習スクリプトを自作する。

- 元のファイル（`go2_env.py` / `go2_train.py` / `go2_backflip.py`）は書き換えず、`%%writefile` で新しいファイルを作る
- 学習も評価も別プロセス（`!python`）で動かすので、`gs.init()` の再起動問題は起きない
- 報酬を変えたら、`go2_backflip_env.py` のセルを実行し直してから学習のセルを実行する

| 時間 | 段階 | 主な報酬 |
|---|---|---|
| 0〜0.5 秒 | 踏み切り | 上向きの速度（`jump`） |
| 0.5〜1.1 秒 | 空中で回転 | 後ろ向きの回転速度（`ang_vel_y`） |
| 1.1〜2.0 秒 | 着地 | 立ち姿勢の高さ（`landing_height`） |
| 全体 | | 時刻ごとの目標の姿勢に近いこと（`orientation_tracking`） |

In [ ]:
%%writefile {LOCO_DIR}/go2_backflip_env.py
import math

import torch

from go2_backflip import get_cfgs as get_backflip_cfgs
from go2_env import Go2Env


def get_cfgs():
    # 環境の設定は公式の go2_backflip.py のもの（PD ゲイン、傾きで打ち切らない など）を流用する
    env_cfg, obs_cfg, _, command_cfg = get_backflip_cfgs()
    env_cfg["episode_length_s"] = 2.0
    reward_cfg = {
        "flip_start_s": 0.5,  # 回転を始める時刻
        "flip_duration_s": 0.6,  # 1 回転にかける時間
        "orientation_sigma": 0.25,
        "max_ang_vel_y": 10.0,  # これ以上速く回っても報酬は増えない [rad/s]
        "base_height_target": 0.3,
        "reward_scales": {
            "orientation_tracking": 5.0,
            "ang_vel_y": 1.0,
            "jump": 2.0,
            "landing_height": 2.0,
            "ang_vel_xz": -0.1,
            "dof_vel": -1e-4,
            "action_rate": -0.01,
        },
    }
    return env_cfg, obs_cfg, reward_cfg, command_cfg


class Go2BackflipEnv(Go2Env):
    def _update_observation(self):
        # 公式の BackflipEnv と同じく、エピソード内の経過時間（phase）を観測に加えて 60 次元にする
        phase = torch.pi * self.episode_length_buf[:, None] / self.max_episode_length
        self.obs_buf = torch.cat(
            [
                self.base_ang_vel * self.obs_scales["ang_vel"],  # 3
                self.projected_gravity,  # 3
                (self.dof_pos - self.default_dof_pos) * self.obs_scales["dof_pos"],  # 12
                self.dof_vel * self.obs_scales["dof_vel"],  # 12
                self.actions,  # 12
                self.last_actions,  # 12
                torch.sin(phase),
                torch.cos(phase),
                torch.sin(phase / 2),
                torch.cos(phase / 2),
                torch.sin(phase / 4),
                torch.cos(phase / 4),
            ],
            dim=-1,
        )

    def _time(self):
        return self.episode_length_buf * self.dt

    def _flip_progress(self):
        # 回転の進み具合（回転前は 0、回転後は 1）
        cfg = self.reward_cfg
        return torch.clamp((self._time() - cfg["flip_start_s"]) / cfg["flip_duration_s"], 0.0, 1.0)

    def _in_flip(self):
        cfg = self.reward_cfg
        t = self._time()
        return ((t >= cfg["flip_start_s"]) & (t < cfg["flip_start_s"] + cfg["flip_duration_s"])).float()

    # Reward functions.
    def _reward_orientation_tracking(self):
        # 時刻ごとの目標の姿勢に近いほど高い。
        # バク転は頭が上がる向き＝体の y 軸まわりに負の回転なので、目標のピッチ角は 0 → -2π
        pitch = -2 * math.pi * self._flip_progress()
        target_gravity = torch.stack([torch.sin(pitch), torch.zeros_like(pitch), -torch.cos(pitch)], dim=-1)
        error = torch.sum(torch.square(self.projected_gravity - target_gravity), dim=1)
        return torch.exp(-error / self.reward_cfg["orientation_sigma"])

    def _reward_ang_vel_y(self):
        # 回転中は後ろ向きの回転速度を評価
        return torch.clamp(-self.base_ang_vel[:, 1], max=self.reward_cfg["max_ang_vel_y"]) * self._in_flip()

    def _reward_jump(self):
        # 踏み切り直前から回転の前半までは、上向きの速度（ワールド座標）を評価
        cfg = self.reward_cfg
        t = self._time()
        active = (t >= cfg["flip_start_s"] - 0.1) & (t < cfg["flip_start_s"] + 0.5 * cfg["flip_duration_s"])
        return torch.clamp(self.robot.get_vel()[:, 2], min=0.0, max=3.0) * active.float()

    def _reward_landing_height(self):
        # 着地後は、立ち姿勢の高さに近いほど高い
        cfg = self.reward_cfg
        landed = self._time() >= cfg["flip_start_s"] + cfg["flip_duration_s"] + 0.2
        height_error = torch.square(self.base_pos[:, 2] - cfg["base_height_target"])
        return torch.exp(-height_error / 0.01) * landed.float()

    def _reward_ang_vel_xz(self):
        # 横回転・ひねりを抑える（ペナルティ）
        return torch.sum(torch.square(self.base_ang_vel[:, [0, 2]]), dim=1)

    def _reward_dof_vel(self):
        # 関節の速すぎる動きを抑える（ペナルティ）
        return torch.sum(torch.square(self.dof_vel), dim=1)

In [ ]:
%%writefile {LOCO_DIR}/go2_backflip_train.py
import argparse
import os
import pickle
import shutil

from rsl_rl.runners import OnPolicyRunner

import genesis as gs

from go2_backflip_env import Go2BackflipEnv, get_cfgs
from go2_train import get_train_cfg


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("-e", "--exp-name", type=str, default="go2-backflip")
    parser.add_argument("-b", "--num-envs", type=int, default=4096)
    parser.add_argument("--max-iterations", type=int, default=1000)
    parser.add_argument("--seed", type=int, default=1)
    args = parser.parse_args()

    log_dir = f"logs/{args.exp_name}"
    env_cfg, obs_cfg, reward_cfg, command_cfg = get_cfgs()
    train_cfg = get_train_cfg(args.exp_name)

    if os.path.exists(log_dir):
        shutil.rmtree(log_dir)
    os.makedirs(log_dir, exist_ok=True)

    with open(f"{log_dir}/cfgs.pkl", "wb") as f:
        pickle.dump([env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg], f)

    gs.init(backend=gs.gpu, precision="32", logging_level="warning", seed=args.seed, performance_mode=True)

    env = Go2BackflipEnv(
        num_envs=args.num_envs, env_cfg=env_cfg, obs_cfg=obs_cfg, reward_cfg=reward_cfg, command_cfg=command_cfg
    )

    runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)

    # 歩行と違い、バク転は毎回エピソードの最初（踏み切り前）から始める必要があるので、開始時刻をばらつかせない
    runner.learn(num_learning_iterations=args.max_iterations, init_at_random_ep_len=False)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {LOCO_DIR}/go2_backflip_record.py
import argparse
import glob
import os
import pickle
import re

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import torch

import genesis as gs


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("-e", "--exp-name", type=str, default="go2-backflip")
    parser.add_argument("--pretrained", action="store_true", help="公式の学習済みモデル（お手本）を使う")
    parser.add_argument("--episodes", type=int, default=3, help="録画するバク転の回数")
    parser.add_argument("-o", "--output", type=str, default="/content/backflip.mp4")
    args = parser.parse_args()

    gs.init(backend=gs.gpu, logging_level="warning")

    if args.pretrained:
        from go2_backflip import BackflipEnv, get_cfgs

        env_cfg, obs_cfg, reward_cfg, command_cfg = get_cfgs()
        env_cfg["episode_length_s"] = 2
        env_cls = BackflipEnv
    else:
        from go2_backflip_env import Go2BackflipEnv

        log_dir = f"logs/{args.exp_name}"
        env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"{log_dir}/cfgs.pkl", "rb"))
        reward_cfg["reward_scales"] = {}
        env_cls = Go2BackflipEnv

    # 環境は内部でシーンを build してしまうので、build の直前に横から見るカメラを追加する
    cams = []
    orig_build = gs.Scene.build

    def build_with_camera(self, *build_args, **build_kwargs):
        cams.append(self.add_camera(res=(640, 480), pos=(0.0, 2.5, 0.6), lookat=(0.0, 0.0, 0.4), fov=40, GUI=False))
        return orig_build(self, *build_args, **build_kwargs)

    gs.Scene.build = build_with_camera
    env = env_cls(
        num_envs=1,
        env_cfg=env_cfg,
        obs_cfg=obs_cfg,
        reward_cfg=reward_cfg,
        command_cfg=command_cfg,
        show_viewer=False,
    )
    gs.Scene.build = orig_build
    cam = cams[0]

    if args.pretrained:
        # go2_backflip.py と同じネットワークを作って重みを読み込む
        policy = torch.nn.Sequential(
            torch.nn.Linear(obs_cfg["num_obs"], 512),
            torch.nn.ELU(),
            torch.nn.Linear(512, 256),
            torch.nn.ELU(),
            torch.nn.Linear(256, 128),
            torch.nn.ELU(),
            torch.nn.Linear(128, env_cfg["num_actions"]),
        )
        policy.load_state_dict(torch.load("backflip/single.pt", weights_only=True))
        policy.to(device=gs.device)
    else:
        from rsl_rl.runners import OnPolicyRunner

        ckpt_path = max(glob.glob(f"{log_dir}/model_*.pt"), key=lambda p: int(re.findall(r"model_([0-9]+)", p)[0]))
        print("load", ckpt_path)
        runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)
        runner.load(ckpt_path)
        policy = runner.get_inference_policy(device=gs.device)

    obs = env.reset()
    cam.start_recording(save_to_filename=args.output, fps=int(1 / env.dt))
    with torch.no_grad():
        # エピソードは max_episode_length ステップを超えると自動でリセットされ、次のバク転が始まる
        for _ in range((env.max_episode_length + 1) * args.episodes):
            obs, _, _, _ = env.step(policy(obs))
    cam.stop_recording()
    print("saved", args.output)


if __name__ == "__main__":
    main()

### 5-1. お手本（公式の学習済みモデル）を見る

In [ ]:
from IPython.display import Video

%cd {LOCO_DIR}
!python go2_backflip_record.py --pretrained -o /content/backflip_pretrained.mp4
Video('/content/backflip_pretrained.mp4', embed=True, width=640)

### 5-2. 学習

まずは少ない回数（100 程度）で動くか確認し、`Mean episode rew_〇〇` の各報酬が増えているかを見てから回数を増やす。

In [ ]:
import shutil

BF_EXP_NAME = 'go2-backflip'   # 設定を変えて学習するときは名前も変える（同じ名前だと前の結果が消える）
BF_NUM_ENVS = 4096
BF_MAX_ITER = 100

%cd {LOCO_DIR}
!python -X faulthandler go2_backflip_train.py -e {BF_EXP_NAME} -b {BF_NUM_ENVS} --max-iterations {BF_MAX_ITER}
!ls {LOCO_DIR}/logs/{BF_EXP_NAME}

if SAVE_DIR:
    shutil.copytree(f'{LOCO_DIR}/logs/{BF_EXP_NAME}', f'{SAVE_DIR}/logs/{BF_EXP_NAME}', dirs_exist_ok=True)
    print('saved to', f'{SAVE_DIR}/logs/{BF_EXP_NAME}')

### 5-3. 評価（学習したモデルでバク転を録画）

In [ ]:
from IPython.display import Video

BF_VIDEO_PATH = f'/content/{BF_EXP_NAME}.mp4'

%cd {LOCO_DIR}
!python go2_backflip_record.py -e {BF_EXP_NAME} -o {BF_VIDEO_PATH}
if SAVE_DIR:
    shutil.copy(BF_VIDEO_PATH, SAVE_DIR)
Video(BF_VIDEO_PATH, embed=True, width=640)